In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p>
  Implement a GPU program that converts an RGB image to grayscale on the GPU.
  Given an input RGB image represented as a 1D array of 32-bit floating point values,
  compute the corresponding grayscale image using the standard RGB to grayscale conversion formula.
</p>

<p>
  The conversion formula is: <code>gray = 0.299 × R + 0.587 × G + 0.114 × B</code>
</p>

<p>
  The input array <code>input</code> contains <code>height × width × 3</code> elements,
  where the RGB values for each pixel are stored consecutively (R, G, B, R, G, B, ...).
  The output array <code>output</code> should contain <code>height × width</code> grayscale values.
</p>

<h2>Implementation Requirements</h2>
<ul>
  <li>External libraries are not permitted</li>
  <li>The <code>solve</code> function signature must remain unchanged</li>
  <li>The final result must be stored in the array <code>output</code></li>
  <li>Use the exact coefficients: 0.299 for red, 0.587 for green, 0.114 for blue</li>
</ul>

<h2>Example 1:</h2>
<pre>
Input:  input = [255.0, 0.0, 0.0, 0.0, 255.0, 0.0, 0.0, 0.0, 255.0, 128.0, 128.0, 128.0], width=2, height=2
Output: output = [76.245, 149.685, 29.07, 128.0]
</pre>

<h2>Example 2:</h2>
<pre>
Input:  input = [100.0, 150.0, 200.0], width=1, height=1
Output: output = [140.75]
</pre>

<h2>Constraints</h2>
<ul>
  <li>1 ≤ <code>width</code> ≤ 4096</li>
  <li>1 ≤ <code>height</code> ≤ 4096</li>
  <li><code>width × height</code> ≤ 4,194,304</li>
  <li>All RGB values are in the range [0.0, 255.0]</li>

  <li>Performance is measured with <code>height</code> = 2,048, <code>width</code> = 2,048</li>
</ul>


# CUDA

In [ ]:
%%writefile solution.cu
#include <cuda_runtime.h>

__global__ void rgb_to_grayscale_kernel(const float* input, float* output, int width, int height) {}

// input, output are device pointers
extern "C" void solve(const float* input, float* output, int width, int height) {
    int total_pixels = width * height;
    int threadsPerBlock = 256;
    int blocksPerGrid = (total_pixels + threadsPerBlock - 1) / threadsPerBlock;

    rgb_to_grayscale_kernel<<<blocksPerGrid, threadsPerBlock>>>(input, output, width, height);
    cudaDeviceSynchronize();
}


# CUTE

In [ ]:
%%writefile solution_cute.py
import cutlass
import cutlass.cute as cute


# input, output are tensors on the GPU
@cute.jit
def solve(input: cute.Tensor, output: cute.Tensor, width: cute.Int32, height: cute.Int32):
    pass


# JAX

In [ ]:
%%writefile solution_jax.py
import jax
import jax.numpy as jnp


# input is a tensor on GPU
@jax.jit
def solve(input: jax.Array, width: int, height: int) -> jax.Array:
    # return output tensor directly
    pass


# MOJO

In [ ]:
%%writefile solution.mojo
from std.gpu.host import DeviceContext
from std.gpu import block_dim, block_idx, thread_idx
from std.memory import UnsafePointer
from std.math import ceildiv


def rgb_to_grayscale_kernel(
    input: UnsafePointer[Float32, MutExternalOrigin],
    output: UnsafePointer[Float32, MutExternalOrigin],
    width: Int32,
    height: Int32,
):
    pass


# input, output are device pointers
@export
def solve(
    input: UnsafePointer[Float32, MutExternalOrigin],
    output: UnsafePointer[Float32, MutExternalOrigin],
    width: Int32,
    height: Int32,
) raises:
    var total_pixels = width * height
    var BLOCK_SIZE: Int32 = 256
    var ctx = DeviceContext()
    var num_blocks = ceildiv(total_pixels, BLOCK_SIZE)

    var _kernel = ctx.compile_function[rgb_to_grayscale_kernel, rgb_to_grayscale_kernel]()
    ctx.enqueue_function(
        _kernel, input, output, width, height, grid_dim=num_blocks, block_dim=BLOCK_SIZE
    )

    ctx.synchronize()


# Torch

In [ ]:
%%writefile solution_pytorch.py
import torch


# input, output are tensors on the GPU
def solve(input: torch.Tensor, output: torch.Tensor, width: int, height: int):
    pass


# Triton

In [ ]:
%%writefile solution_triton.py
import torch
import triton
import triton.language as tl


@triton.jit
def rgb_to_grayscale_kernel(input, output, width, height, BLOCK_SIZE: tl.constexpr):
    pass


# input, output are tensors on the GPU
def solve(input: torch.Tensor, output: torch.Tensor, width: int, height: int):
    total_pixels = width * height
    BLOCK_SIZE = 1024
    grid = (triton.cdiv(total_pixels, BLOCK_SIZE),)
    rgb_to_grayscale_kernel[grid](input, output, width, height, BLOCK_SIZE)


# Evaluate Setup

In [ ]:
# Download required files from GitHub
!mkdir -p core
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/core/challenge_base.py -O core/challenge_base.py
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/core/evaluator.py -O core/evaluator.py
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/easy/66_rgb_to_grayscale/challenge.py -O challenge.py

from challenge import Challenge
from core.evaluator import Evaluate

ch = Challenge()


# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch, EVAL_LANG)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
